# Eval_harness

Imports

In [4]:
import numpy as np
import json
from typing import List, Set, Dict

Golden Set (the data)

In [5]:
golden_set = [
    {
        "question": "If I file a claim, how long until I get paid?",
        "ideal_answer": "Most claims are processed within 30 days.",
        "chunk_ids": [3, 7, 12]
    },
    {
        "question": "What's my deductible?",
        "ideal_answer": "$500 for collision, $250 for comprehensive.",
        "chunk_ids": [2, 5]
    },
    {
        "question": "Am I covered for theft?",
        "ideal_answer": "Yes, theft is covered under comprehensive coverage.",
        "chunk_ids": [1, 6, 9]
    },
    {
        "question": "What should I do if my car breaks down?",
        "ideal_answer": "Call roadside assistance immediately; they'll tow your car to a repair shop.",
        "chunk_ids": [4, 8, 11]
    },
    {
        "question": "How do I file a claim online?",
        "ideal_answer": "Log into your account, click 'File Claim', and upload photos of the damage.",
        "chunk_ids": [5, 10]
    }
]

print(f"Golden set loaded: {len(golden_set)} examples")

Golden set loaded: 5 examples


Retrieval Metrics Function

In [6]:
def compute_retrieval_metrics(retrieved_chunk_ids, relevant_chunk_ids, k=10):
    """
    Compute precision@k, recall@k, MRR, and NDCG@k from scratch.

    Args:
        retrieved_chunk_ids: list of chunk IDs in rank order (e.g., [5, 3, 8, 12, 1, ...])
        relevant_chunk_ids: set of chunk IDs known to be relevant (e.g., {3, 5, 7, 12})
        k: top-k to evaluate (default 10)

    Returns:
        dict with precision@k, recall@k, mrr, ndcg@k
    """

    # Truncate to top-k
    retrieved_chunk_ids = retrieved_chunk_ids[:k]
    relevant_chunk_ids = set(relevant_chunk_ids)

    # Find which retrieved chunks are relevant
    relevant_in_top_k = [cid in relevant_chunk_ids for cid in retrieved_chunk_ids]
    num_relevant_in_top_k = sum(relevant_in_top_k)

    # Precision@k: purity of top-k
    precision_k = num_relevant_in_top_k / k

    # Recall@k: coverage of all relevant chunks
    recall_k = num_relevant_in_top_k / len(relevant_chunk_ids) if len(relevant_chunk_ids) > 0 else 0

    # MRR (Mean Reciprocal Rank): 1 / rank of first relevant chunk
    mrr = 0
    for rank, is_relevant in enumerate(relevant_in_top_k, start=1):
        if is_relevant:
            mrr = 1.0 / rank
            break

    # NDCG@k: ranking quality (normalized discounted cumulative gain)
    # DCG = sum of (relevance / log2(rank + 1))
    dcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(relevant_in_top_k, start=1))

    # IDCG = DCG of perfect ranking (all relevant first, then non-relevant)
    num_relevant_total = len(relevant_chunk_ids)
    ideal_ranking = [1] * min(num_relevant_total, k) + [0] * (k - min(num_relevant_total, k))
    idcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(ideal_ranking, start=1))

    # NDCG = DCG / IDCG
    ndcg_k = dcg / idcg if idcg > 0 else 0

    return {
        "precision@k": precision_k,
        "recall@k": recall_k,
        "mrr": mrr,
        "ndcg@k": ndcg_k
    }

print("Retrieval metrics function defined ✓")

Retrieval metrics function defined ✓


Test Example

In [7]:
# Simulate a retrieval result
retrieved = [5, 3, 8, 12, 1, 9, 2, 6, 4, 10]  # rank order
relevant = {3, 5, 7, 12}  # ground truth relevant chunks

metrics = compute_retrieval_metrics(retrieved, relevant, k=10)

print("Test Example:")
print(f"  Retrieved (top 10): {retrieved}")
print(f"  Relevant (from golden set): {relevant}")
print(f"  Precision@10: {metrics['precision@k']:.3f}")
print(f"  Recall@10: {metrics['recall@k']:.3f}")
print(f"  MRR: {metrics['mrr']:.3f}")
print(f"  NDCG@10: {metrics['ndcg@k']:.3f}")

Test Example:
  Retrieved (top 10): [5, 3, 8, 12, 1, 9, 2, 6, 4, 10]
  Relevant (from golden set): {3, 12, 5, 7}
  Precision@10: 0.300
  Recall@10: 0.750
  MRR: 1.000
  NDCG@10: 0.805


Evaluate on Golden Set

In [8]:
# For each golden example, compute metrics
# (Pretend your retriever returns this result)

results = []
for i, golden_entry in enumerate(golden_set):
    # Simulate different retrieval results for each query
    if i == 0:
        retrieved = [3, 5, 8, 12, 1, 9, 2, 6, 4, 10]
    elif i == 1:
        retrieved = [2, 5, 7, 1, 3, 9, 4, 6, 8, 10]
    elif i == 2:
        retrieved = [1, 6, 9, 3, 5, 2, 7, 12, 4, 8]
    else:
        retrieved = list(range(1, 11))

    relevant = golden_entry["chunk_ids"]
    metrics = compute_retrieval_metrics(retrieved, relevant, k=10)

    results.append({
        "question": golden_entry["question"],
        "metrics": metrics
    })

    print(f"Q{i+1}: {golden_entry['question'][:50]}...")
    print(f"  P@10: {metrics['precision@k']:.3f} | R@10: {metrics['recall@k']:.3f} | MRR: {metrics['mrr']:.3f} | NDCG: {metrics['ndcg@k']:.3f}")
    print()

print("=" * 60)
print(f"Baseline Metrics Across {len(golden_set)} Examples:")
print("=" * 60)
avg_precision = np.mean([r["metrics"]["precision@k"] for r in results])
avg_recall = np.mean([r["metrics"]["recall@k"] for r in results])
avg_mrr = np.mean([r["metrics"]["mrr"] for r in results])
avg_ndcg = np.mean([r["metrics"]["ndcg@k"] for r in results])

print(f"Avg Precision@10: {avg_precision:.3f}")
print(f"Avg Recall@10: {avg_recall:.3f}")
print(f"Avg MRR: {avg_mrr:.3f}")
print(f"Avg NDCG@10: {avg_ndcg:.3f}")

Q1: If I file a claim, how long until I get paid?...
  P@10: 0.200 | R@10: 0.667 | MRR: 1.000 | NDCG: 0.671

Q2: What's my deductible?...
  P@10: 0.200 | R@10: 1.000 | MRR: 1.000 | NDCG: 1.000

Q3: Am I covered for theft?...
  P@10: 0.300 | R@10: 1.000 | MRR: 1.000 | NDCG: 1.000

Q4: What should I do if my car breaks down?...
  P@10: 0.200 | R@10: 0.667 | MRR: 0.250 | NDCG: 0.350

Q5: How do I file a claim online?...
  P@10: 0.200 | R@10: 1.000 | MRR: 0.200 | NDCG: 0.414

Baseline Metrics Across 5 Examples:
Avg Precision@10: 0.220
Avg Recall@10: 0.867
Avg MRR: 0.690
Avg NDCG@10: 0.687
